In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import sys
sys.path.insert(0, "/Users/jihaein/Desktop/fds_main/investment-evidence-system")

In [2]:
from backtest.engine import load_daily_price
from backtest.montecarlo import run_montecarlo
import os; from dotenv import load_dotenv; load_dotenv()

base = os.getenv("BACKTEST_DATA_DIR")
grid = load_daily_price()
detail, summary = run_montecarlo(grid, base, n_pick=20, n_runs=10, seed=0)
print(summary)
print(detail.groupby("strategy")[["누적수익률","MDD"]].agg(["min","mean","max"]))

             누적수익률   수익률_중앙값       MDD   Sortino    Sharpe   평균_현금대체  평균_매도수  \
strategy                                                                       
무손절       0.438450  0.380739  0.514163  0.537186  0.394163       NaN     0.0   
단순손절     -0.003649 -0.067181  0.296231  0.007188  0.000747 -0.608356    17.7   
서서히       0.345384  0.254256  0.439094  0.489910  0.358703 -0.367674     5.2   

          win_vs_무손절  
strategy              
무손절              0.0  
단순손절             0.0  
서서히              0.3  
             누적수익률                           MDD                    
               min      mean       max       min      mean       max
strategy                                                            
단순손절     -0.165917 -0.003649  0.459766  0.232975  0.296231  0.429493
무손절       0.006687  0.438450  0.905283  0.436034  0.514163  0.582225
서서히      -0.020157  0.345384  0.758285  0.348303  0.439094  0.487389


In [3]:
from backtest.montecarlo import _verdict, DEFAULT_BASELINE_DATE
from models.seoseohi.pipeline import load_ksic_from_base
from models.seoseohi.fscore import list_snapshot_dates

ksic = load_ksic_from_base(base)
dates = [d for d in list_snapshot_dates(base) if d > DEFAULT_BASELINE_DATE]
for s in grid.columns:
    for d in dates:
        _verdict(base, ksic, s, DEFAULT_BASELINE_DATE, d)
        
print("cache warmed:", len(__import__('backtest.montecarlo', fromlist=['_VERDICT_CACHE'])._VERDICT_CACHE))

cache warmed: 2280


In [4]:
d2, s2 = run_montecarlo(grid, base, years=2, seed=0)
d4, s4 = run_montecarlo(grid, base, years=4, seed=0)
print("=== 2년 ==="); print(s2.round(3))
print("=== 4년 ==="); print(s4.round(3))

=== 2년 ===
          누적수익률  수익률_중앙값    MDD  Sortino  Sharpe  평균_현금대체  평균_매도수  win_vs_무손절
strategy                                                                     
무손절       0.144    0.129  0.224    0.597   0.426      NaN     0.0         0.0
단순손절      0.056    0.058  0.181    0.338   0.234   -0.145    13.5         0.1
서서히       0.145    0.126  0.217    0.610   0.435   -0.011     2.7         0.6
=== 4년 ===
          누적수익률  수익률_중앙값    MDD  Sortino  Sharpe  평균_현금대체  평균_매도수  win_vs_무손절
strategy                                                                     
무손절       0.108    0.084  0.388    0.286   0.207      NaN     0.0         0.0
단순손절     -0.049   -0.060  0.261   -0.079  -0.062   -0.230    15.9         0.2
서서히       0.088    0.100  0.354    0.263   0.193   -0.077     4.6         0.7


## Result Interpretation

#### 2-year window (2016->2018) --> 서서히랑 무손절 거의 비김
- 서서히 and 무손절 basically tied (Return: 9.3% vs 9.2% · Sortino: 0.456 vs 0.453 · Sharpe: 0.305 vs 0.306 — a dead heat on return and risk-adjusted terms.)
- BUT 서서 has lower draw down (MDD 26.3% vs 27.5%) and wins the head-to-head in 60% of baskets (win_vs_무손절 0.6).
- So over 2 years: matches buy-and-hold, with a slightly smoother ride and a small majority-of-baskets edge.

#### 4-year window (2016->2020) --> 서서히가 조금의 차이로 이김
- Return: 1.7% vs 1.5%, and median 3.8% vs 0.8% (the median gap is the real tell — buy-and-hold’s typical basket barely broke even).
- Lower drawdown (40.3% vs 43.7%) and wins 70% of baskets (win_vs_무손절 0.7).
- Risk-adjusted (Sortino 0.175 vs 0.183) is a near-tie, so call it: 서서히 ≥ hold over 4 years, with less pain.


# 어디서부터 잘못됐을까....

In [5]:
import pandas as pd
from models.seoseohi.pipeline import run_seoseohi_pit, load_ksic_from_base
from models.seoseohi.fscore import list_snapshot_dates

baseline = "2016-04-01"
horizon  = 500 # 2 years
ksic = load_ksic_from_base(base)
data = {"base_dir": base, "ksic": ksic}

eval_dates = [d for d in list_snapshot_dates(base) if pd.Timestamp(d) > pd.Timestamp(baseline)]

rows = []
for eval_date in eval_dates:
    days = grid.index[grid.index >= pd.Timestamp(eval_date)]
    if len(days) < horizon + 1:          # 앞으로 120일이 안 남으면 건너뜀
        continue
    window = days[:horizon + 1]
    for s in grid.columns:
        verdict = run_seoseohi_pit(data, {"stock_code": s, "buy_date": baseline}, eval_date)["verdict"]
        fwd = (1 + grid.loc[window, s].fillna(0) / 100).prod() - 1
        rows.append({"eval_date": eval_date, "stock": s, "verdict": verdict, "fwd_return": fwd})

pool = pd.DataFrame(rows)
print("샘플 수:", len(pool))
print(pool.groupby("verdict")["fwd_return"].agg(["count", "mean", "median"]))

샘플 수: 1320
            count      mean    median
verdict                              
No            598  0.146563 -0.105363
Yes(훼손 우려)     89  0.061858 -0.129765
유보            179 -0.052423 -0.044739
주의            454  0.076103 -0.151819


In [6]:
import sys, shutil
for m in [k for k in list(sys.modules) if k.startswith("backtest")]:
    del sys.modules[m]
shutil.rmtree("/Users/jihaein/Desktop/fds_main/investment-evidence-system/backtest/__pycache__",
              ignore_errors=True)

from backtest.engine import load_daily_price
from backtest.montecarlo import run_montecarlo
import os; from dotenv import load_dotenv; load_dotenv()
base = os.getenv("BACKTEST_DATA_DIR")
grid = load_daily_price()

# Phase 2 — 서서히 매도 품질 스코어카드 (게이트 OFF = 현재 모델)
for yrs in [2, 4]:
    d, s = run_montecarlo(grid, base, n_pick=20, n_runs=10, seed=0, years=yrs)
    ss = d[d.strategy == "서서히"]
    good, bad = int(ss["좋은매도"].sum()), int(ss["성급한매도"].sum())
    total = good + bad
    rate = good / total if total else float("nan")
    print(f"=== {yrs}년 ===  총매도 {total}  |  좋은매도 {good} / 성급한매도 {bad}  |  "
          f"적중률 {rate:.1%}  |  평균B {ss['평균_현금대체'].mean():.3f}")

=== 2년 ===  총매도 27  |  좋은매도 15 / 성급한매도 12  |  적중률 55.6%  |  평균B -0.011
=== 4년 ===  총매도 46  |  좋은매도 33 / 성급한매도 13  |  적중률 71.7%  |  평균B -0.077


In [7]:
import sys, shutil
for m in [k for k in list(sys.modules) if k.startswith("backtest")]:
    del sys.modules[m]
shutil.rmtree("/Users/jihaein/Desktop/fds_main/investment-evidence-system/backtest/__pycache__",
              ignore_errors=True)

from backtest.engine import load_daily_price
from backtest.montecarlo import sell_study
import os, pandas as pd; from dotenv import load_dotenv; load_dotenv()
base = os.getenv("BACKTEST_DATA_DIR")
grid = load_daily_price()

sells = sell_study(grid, base, n_pick=20, n_runs=10, seed=0, years=4)
uniq = sells.drop_duplicates(["stock", "sell_date"])        # 고유 매도 결정만
print(f"고유 매도 결정: {len(uniq)}건\n")

# 1) 판 뒤 주가가 어떻게 됐나 (R_hold = 안 팔고 들고 있었으면 났을 수익률)
buckets = pd.cut(uniq["R_hold"], [-1, -0.5, -0.3, -0.1, 0.1, 0.3, 1, 99],
                 labels=["≤-50%","-50~-30%","-30~-10%","-10~+10%","+10~+30%","+30~+100%",">+100%"])
print("판 뒤 주가 분포:")
print(buckets.value_counts().sort_index(), "\n")

# 2) 재앙 회피 — 판 뒤 -30% 이하로 폭락한 것 (= 잘 팔았다)
caught = uniq[uniq.R_hold <= -0.3].sort_values("R_hold")
print(f"[재앙 회피] 판 뒤 -30%↓ : {len(caught)}건")
print(caught[["stock","sell_date","R_hold"]].head(10).to_string(index=False), "\n")

# 3) 놓친 반등 — 판 뒤 +30% 이상 오른 것 (= 성급히 팔았다)
missed = uniq[uniq.R_hold >= 0.3].sort_values("R_hold", ascending=False)
print(f"[놓친 반등] 판 뒤 +30%↑ : {len(missed)}건")
print(missed[["stock","sell_date","R_hold"]].head(10).to_string(index=False), "\n")

print(f"재앙(-30%↓) 회피 {len(caught)}건  vs  반등(+30%↑) 오판 {len(missed)}건")
print(f"최고의 회피: {uniq.R_hold.min():+.1%}   |   최악의 오판: {uniq.R_hold.max():+.1%}")

고유 매도 결정: 25건

판 뒤 주가 분포:
R_hold
≤-50%        3
-50~-30%     2
-30~-10%     5
-10~+10%     7
+10~+30%     4
+30~+100%    3
>+100%       1
Name: count, dtype: int64 

[재앙 회피] 판 뒤 -30%↓ : 5건
 stock  sell_date    R_hold
137940 2017-06-01 -0.586514
038870 2017-04-03 -0.531808
051780 2016-09-01 -0.513017
131760 2017-12-01 -0.482906
040160 2016-12-01 -0.302241 

[놓친 반등] 판 뒤 +30%↑ : 4건
 stock  sell_date   R_hold
078130 2016-12-01 8.998075
086390 2016-09-01 0.532383
052710 2018-12-03 0.492171
014970 2018-06-01 0.420708 

재앙(-30%↓) 회피 5건  vs  반등(+30%↑) 오판 4건
최고의 회피: -58.7%   |   최악의 오판: +899.8%


In [8]:
import pandas as pd, os
from backtest.montecarlo import sell_study

# 종목명
nm = pd.read_csv(os.path.join(base, "12_기업개황_KSIC.csv"),
                 header=0, skiprows=[1], dtype=str, encoding="utf-8-sig")
nm["stock_code"] = nm["stock_code"].str.zfill(6)
name_map = dict(zip(nm["stock_code"], nm["corp_name"]))

sells = sell_study(grid, base, n_pick=20, n_runs=10, seed=0, years=4)
uniq = sells.drop_duplicates(["stock", "sell_date"]).copy()

# 매수(2016-01-04) ~ 매도 시점까지의 누적수익률
entry = pd.Timestamp("2016-01-04")
def r_until(row):
    days = grid.loc[entry:row["sell_date"], row["stock"]].dropna()
    return (1 + days.iloc[1:] / 100).prod() - 1 if len(days) > 1 else float("nan")
uniq["매도시점_누적"] = uniq.apply(r_until, axis=1)

uniq["종목명"]    = uniq["stock"].map(name_map)
uniq["판정"]      = uniq["good"].map({True: "✅ 좋은매도", False: "❌ 성급한매도"})
uniq["매수~매도"] = uniq["매도시점_누적"].map(lambda x: f"{x:+.1%}")
uniq["매도후"]    = uniq["R_hold"].map(lambda x: f"{x:+.1%}")
uniq["매도일"]    = pd.to_datetime(uniq["sell_date"]).dt.date
uniq = uniq.sort_values("매도일")

print(uniq[["매도일","stock","종목명","매수~매도","판정","매도후"]].to_string(index=False))

       매도일  stock        종목명  매수~매도      판정     매도후
2016-09-01 086390   (주)유니테스트  +8.0% ❌ 성급한매도  +53.2%
2016-09-01 070300    (주)퀀텀레일  +2.8%  ✅ 좋은매도   +1.0%
2016-09-01 051780   (주)큐로홀딩스 -21.3%  ✅ 좋은매도  -51.3%
2016-09-01 083310   (주)엘오티베큠 +74.6%  ✅ 좋은매도  -29.4%
2016-09-01 054780    (주)키이스트 -18.9%  ✅ 좋은매도  -12.4%
2016-12-01 138690   (주)엘아이에스 -56.1% ❌ 성급한매도  +25.2%
2016-12-01 078130   에스엠국일(주)  -0.7% ❌ 성급한매도 +899.8%
2016-12-01 040160   (주)누리플렉스 -39.3%  ✅ 좋은매도  -30.2%
2017-04-03 038870  에코심플렉스(주) -51.7%  ✅ 좋은매도  -53.2%
2017-04-03 093520     (주)매커스 +25.5%  ✅ 좋은매도  -13.5%
2017-04-03 073570   (주)리튬포어스 -35.2% ❌ 성급한매도  +10.9%
2017-06-01 137940   (주)넥스트아이 +61.3%  ✅ 좋은매도  -58.7%
2017-06-01 098660      ㈜에스티오  -9.5%  ✅ 좋은매도   -4.3%
2017-12-01 131760     (주)파인텍  -9.9%  ✅ 좋은매도  -48.3%
2018-06-01 014970    삼륭물산(주) +24.7% ❌ 성급한매도  +42.1%
2018-12-03 052710     (주)아모텍  -3.9% ❌ 성급한매도  +49.2%
2018-12-03 090710    휴림로봇(주) +31.7%  ✅ 좋은매도   -5.4%
2019-04-01 123040  (주)엠에스오토텍 +35.8% ❌ 성급한매도  +18.4%
2019-06-03 0